# Getting Started with `bayespecon`

This notebook walks through a complete spatial econometric workflow:

1. Simulate spatial data from a known DGP
2. Fit a baseline OLS model
3. Run Bayesian LM diagnostics to test for spatial dependence
4. Use the decision tree to select a better model
5. Fit the recommended SAR model
6. Compute direct, indirect, and total spatial effects
7. Peek at the underlying PyMC model for customization

In [ ]:
import libpysal

from bayespecon.dgp import simulate_sar

gdf = simulate_sar(n=20, beta=[1, 0.4, 2.5], rho=0.6, create_gdf=True)

In [ ]:
gdf

In [ ]:
gdf.plot("X_1").set_title("X1")

In [ ]:
gdf.plot("y").set_title("y")

## 1. Create a Spatial Weights Matrix

Spatial econometric models require a spatial weights matrix $W$ that encodes the spatial relationships between observations. Here we use Rook contiguity (regions that share a boundary or vertex are neighbors).

In [ ]:
G = libpysal.graph.Graph.build_contiguity(gdf, rook=False).transform("r")

## 2. Fit a Baseline OLS Model

We start with a non-spatial OLS regression to establish a baseline.

$$ y = X\beta + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I) $$

In [ ]:
# remove the intercept since we have one (-1)
form = "y ~ -1 + X_0 + X_1 + X_2"

In [ ]:
from bayespecon import OLS

ols = OLS(formula=form, W=G, data=gdf)
ols.fit(draws=1000, tune=500, chains=2, random_seed=42)
ols.summary()

### Use `arviz` to inspect fit

the `fit` method attaches an `inference_data` object to each model that can be used directly with `arviz` functions

In [ ]:
import arviz as az

az.plot_forest(ols.inference_data)

In [ ]:
az.plot_trace(ols.inference_data)

## 3. Run Bayesian LM Diagnostics

The `spatial_diagnostics()` method runs a battery of Bayesian LM tests (Doğan, Taşpınar \& Bera 2021) that check whether spatial dependence is present in the residuals.

- **LM-Lag**: Tests $H_0: \rho = 0$ (no spatial lag on $y$)
- **LM-Error**: Tests $H_0: \lambda = 0$ (no spatial error correlation)
- **LM-SDM-Joint**: Joint test for $\rho = 0$ and $\gamma = 0$
- **Robust-LM-Lag / Robust-LM-Error**: Neyman-orthogonal robust versions

In [ ]:
diag = ols.spatial_diagnostics()
diag

## 4. Model Selection Decision Tree

The `spatial_diagnostics_decision()` method walks a Koley \& Bera decision tree using the Bayesian p-values above and recommends the next model to fit.

In [ ]:
decision = ols.spatial_diagnostics_decision(alpha=0.05, format="ascii")
print(decision)

In [ ]:
ols.spatial_diagnostics_decision(alpha=0.05)

## 5. Fit the Recommended SAR Model

If the diagnostics suggest spatial dependence, we fit a Spatial Autoregressive (SAR) model:

$$ y = \rho W y + X\beta + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I) $$

The likelihood includes the Jacobian $\log|I - \rho W|$ so that posterior inference on $\rho$ is exact.

In [ ]:
from bayespecon import SAR

sar = SAR(formula=form, W=G, data=gdf)
sar.fit(draws=2000, tune=100, chains=4, random_seed=42)
sar.summary()

In [ ]:
az.plot_trace(sar.inference_data)

## 6. Compute Spatial Effects

For the SAR model, a change in $X$ propagates through the spatial multiplier $(I - \rho W)^{-1}$. The effects decompose into:

- **Direct**: Effect of $X_i$ on $y_i$ (own-region)
- **Indirect**: Effect of $X_j$ on $y_i$ for $j \neq i$ (spillover)
- **Total**: Direct + Indirect

In [ ]:
effects = sar.spatial_effects()
effects

## 7. Access the PyMC Model for Customization

Every `bayespecon` model compiles to a `pymc.Model` object. You can inspect it, add custom variables, or re-sample with different settings.

In [ ]:
# The PyMC model is available as a property
pm_model = sar.pymc_model
print(pm_model)

# List all named variables
print("Named vars:", list(pm_model.named_vars.keys()))

## Summary

In this quickstart we:

1. Loaded spatial data and built a queen contiguity weights matrix
2. Fit a baseline OLS model
3. Ran Bayesian LM diagnostics to detect spatial dependence
4. Used the decision tree to recommend a SAR model
5. Fit the SAR model and computed spatial effects
6. Accessed the underlying PyMC model for further customization

For more advanced topics (panel models, flow models, negative binomial counts, robust regression, bridge sampling for Bayes factors), see the other notebooks in this guide.